# 1주차 보충 · 라벨이 일부만 있는 데이터의 분류

IOAI 2025 개인전 *Antique Painting Authentication*은 그림의 디지털 특성으로 진품과 복제품을 구분하는 문제입니다. 이 문제에서 라벨은 진품이나 복제품으로 확인된 결과를 나타내는 값입니다. 공개 훈련 데이터 500점 중 진품으로 확인된 그림은 2점, 복제품으로 확인된 그림은 2점입니다. 나머지 496점에도 특성값은 있지만 정답은 알려지지 않았습니다. [대회 훈련 데이터](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Antique/training_set/training_set.csv)

1–3절에서는 진위가 확인된 그림으로 진품과 복제품의 **평균 벡터**를 구합니다. 평균 벡터는 같은 종류의 그림을 모아 특성마다 평균을 낸 뒤 그 값들을 차례로 적은 것입니다. 새 그림을 두 평균 벡터 중 어느 쪽에 더 가까운지로 분류하고, 학습에 쓰지 않은 그림으로 정확도를 구합니다.

[4절의 독립 문제](#pipeline-device)에서는 다른 표로 같은 원리를 직접 적용합니다. 파일을 저장하면서 행 순서가 달라지는 경우는 [선택 실습](#pipeline-submit)에서 다룹니다.


<a id="pipeline-contract"></a>

## 1. 그림의 특성, 라벨, 훈련 데이터

대회 훈련 파일에서 한 행은 그림 한 점입니다. <code>feature1</code>부터 <code>feature5</code>까지는 **특성**, 즉 그림을 나타내는 다섯 개의 수입니다. 원문은 각 특성이 어떤 물리량인지 밝히지 않습니다. 마지막 열 <code>Authenticated</code>는 **라벨**, 즉 확인된 분류 결과를 기록하는 열입니다. 1은 진품, −1은 복제품, 0은 미확인입니다. [공식 문제 1–2절](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Antique/Antique.ipynb)

| 훈련 파일의 행 번호 | 다섯 특성의 값(소수 셋째 자리까지 반올림) | Authenticated |
|---:|---|---:|
| 1 | [0.600, 0.012, 0.209, 1.088, 4.088] | 0 |
| 47 | [−0.534, 4.897, 0.658, 1.269, 3.627] | −1 |
| 142 | [3.188, −2.940, −0.404, 0.737, 3.842] | 1 |

행 번호는 열 이름을 제외하고 첫 그림부터 센 순서입니다. 47번째 그림은 복제품의 예로 학습할 수 있습니다. 첫 번째 그림은 다섯 특성이 모두 있어도 진품인지 복제품인지 모릅니다. 미확인 496점을 모두 복제품으로 처리하면 확인하지 않은 정답 496개를 훈련 데이터에 넣게 됩니다. [공개 훈련 CSV](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Antique/training_set/training_set.csv)

**훈련 데이터**는 분류 규칙을 정하는 데 쓰고, **검증 데이터**는 그 규칙으로 새 표본을 얼마나 잘 분류하는지 살펴보는 데 씁니다. **테스트 데이터**는 마지막 평가에 씁니다. Antique의 검증·테스트 데이터에는 각각 별도의 그림 500점이 있습니다. 참가자가 직접 볼 수 있는 것은 훈련 데이터입니다. 채점할 때 제출한 코드가 검증·테스트의 다섯 특성을 읽고 각 그림의 라벨 −1 또는 1을 예측합니다.

모든 대회 문제가 훈련 데이터를 제공하는 것은 아닙니다. IOAI 2026의 *Potato*에는 라벨이 붙은 훈련 데이터가 없습니다. 심판이 알려 주는 두 단어의 비교 결과를 바탕으로 다음 단어를 제안하며, 공개된 게임으로 제안 방법을 시험할 수 있습니다. 그림의 정답으로 분류기를 학습하는 Antique와 어떤 자료가 다른지 구분해 보세요. [Potato 원문의 Task·Dataset](https://github.com/IOAI-official/IOAI-2026/blob/e1963801aa4cf33a5a03520b39fa6620a7894c54/Individual-Contest/3_Potato/statement.md)


### 계산에 사용할 훈련 표

다음 표는 손으로 계산할 수 있도록 **새로 만든 연습 데이터**입니다. 대회 파일에서 뽑은 값이 아닙니다. 두 특성 <code>feature1</code>과 <code>feature2</code>는 같은 척도의 숫자로 두고, <code>label</code>에는 대회와 같은 기호를 씁니다. <code>row_id</code>는 같은 그림을 다시 찾기 위해 붙인 <strong>식별자(ID)</strong>입니다. 예를 들어 T2는 이 훈련 표의 두 번째 그림입니다. 대회 훈련 파일에는 별도의 식별자 열이 없습니다.

라벨이 확인된 표본과 정답을 짝지어 분류 규칙을 배우는 방법을 **지도학습**이라고 합니다. 여기서는 라벨이 있는 훈련 그림만 사용합니다. 미확인 그림의 특성값까지 활용하도록 한 대회의 전체 요구 중 일부를 연습하는 것입니다.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

feature_cols = ["feature1", "feature2"]
training = pd.DataFrame([
    ("T1", 1, 1, -1), ("T2", 3, np.nan, -1), ("T3", 7, 3, 1),
    ("T4", 9, 5, 1), ("T5", 6, 3, 0),
], columns=["row_id", "feature1", "feature2", "label"])
known_train = training[training["label"].isin([-1, 1])].copy()
display(training)
print("라벨이 확인된 훈련 그림의 ID:", known_train["row_id"].tolist())


,row_id,feature1,feature2,label
0,T1,1,1.0,-1
1,T2,3,NaN,-1
2,T3,7,3.0,1
3,T4,9,5.0,1
4,T5,6,3.0,0


라벨이 확인된 훈련 그림의 ID: ['T1', 'T2', 'T3', 'T4']


## 2. 클래스별 평균 벡터와 최근접 중심 분류기

**클래스**는 분류할 종류를 뜻합니다. 여기에는 진품과 복제품 두 클래스가 있습니다. 연습 표에서 T2의 둘째 특성은 비어 있습니다(<code>NaN</code>). 라벨이 확인된 훈련 그림 중 이 특성을 관측한 T1, T3, T4의 평균 (1+3+5)/3=3으로 빈칸을 채웁니다. 같은 방식으로 첫째 특성의 평균은 (1+3+7+9)/4=5입니다. 검증 그림에 빈칸이 있어도 이 훈련 평균을 씁니다.

복제품 T1과 T2의 특성 벡터는 이제 [1,1]과 [3,3]입니다. 첫째 특성의 평균도 (1+3)/2=2, 둘째 특성의 평균도 2이므로 복제품의 **평균 벡터**, 곧 이 클래스의 **중심**은 [2,2]입니다.

### 왜 평균을 중심으로 삼을까?

첫째 특성의 값 1과 3을 대표할 수 하나를 <code>a</code>라고 합시다. 각 값과 <code>a</code> 사이의 차이를 제곱해 더하면 (1−a)²+(3−a)²입니다. 이 식은 **2(a−2)²+2**와 같습니다. 제곱은 0 이상이므로 a가 2일 때 합이 2로 가장 작습니다.

이 성질은 표본이 늘어나도 같습니다. 한 특성의 훈련값을 x₁,…,xₙ, 그 평균을 m이라고 쓰겠습니다. Σ는 n개 항을 더한다는 기호입니다. xᵢ−a를 (xᵢ−m)+(m−a)로 나누어 제곱하면 다음 식을 얻습니다.

**Σ(xᵢ−a)² = Σ(xᵢ−m)² + n(m−a)²**

각 훈련값 xᵢ에서 그 클래스의 평균 m을 뺀 값을 편차라고 합니다. 편차의 합 Σ(xᵢ−m)이 0이므로 전개할 때 생기는 항 2(m−a)Σ(xᵢ−m)도 0입니다. 오른쪽 첫 항은 a와 관계없고 둘째 항은 a=m일 때 가장 작습니다. 두 점 사이의 제곱거리는 각 좌표에서 두 값의 차이를 구하고, 그 차이를 제곱한 뒤 모두 더한 값입니다. 따라서 **각 특성의 평균을 모은 벡터가 같은 클래스 안에서 제곱거리의 합을 가장 작게 합니다.**


### 새 그림을 분류하는 방법

**최근접 중심 분류기**는 새 그림에서 각 클래스 중심까지의 거리를 비교해 더 가까운 클래스의 라벨을 예측합니다. 두 점 [x,y]와 [a,b] 사이 거리의 제곱은 (x−a)²+(y−b)²입니다. 제곱근을 취해도 대소 관계가 같으므로 제곱거리만 비교해도 됩니다. 한 특성의 단위만 바꿔 값을 10배로 쓰면 그 특성이 제곱거리에 더하는 값은 100배가 됩니다. 그래서 이 연습에서는 두 특성의 척도를 같게 두었습니다.

특성값이 [6,3]인 그림에서 복제품 중심까지의 제곱거리는 (6−2)²+(3−2)²=17입니다. **T3와 T4를 이용해 진품 중심을 먼저 구하세요.** 진품 중심까지의 제곱거리는 얼마이고 어느 라벨을 예측하나요?

두 중심을 비교하려면 두 클래스에 모두 라벨이 확인된 훈련 그림이 있어야 합니다. 빈칸을 채우려면 각 특성에 적어도 하나의 관측값이 필요합니다.

아래 코드에서 <code>values.mean()</code>은 각 특성 열의 관측값으로 평균을 구하고, <code>fillna</code>는 그 평균으로 같은 열의 빈칸을 채웁니다. 이어서 <code>groupby</code>로 라벨이 같은 훈련 행을 묶어 클래스마다 평균을 구합니다. 두 번째 함수의 <code>axis=1</code>은 한 그림의 특성별 제곱 차이를 가로로 더한다는 뜻입니다. 결과 표에서 한 행은 그림 한 점, 한 열은 클래스 하나입니다.


In [2]:
def training_averages(labeled_rows, columns):
    values = labeled_rows[columns]
    if labeled_rows["label"].nunique() < 2:
        raise ValueError("두 클래스 모두 라벨이 확인된 훈련 그림이 필요합니다.")
    if np.isinf(values.to_numpy(dtype=float)).any():
        raise ValueError("관측값은 유한한 수여야 합니다.")
    means = values.mean()
    if means.isna().any():
        raise ValueError("관측값이 없는 훈련 열은 평균을 구할 수 없습니다.")
    completed = values.fillna(means)
    centers = completed.groupby(labeled_rows["label"], sort=True).mean()
    return means, centers

def squared_distances(rows, means, centers):
    completed = rows.loc[:, means.index].fillna(means)
    if not np.isfinite(completed.to_numpy(dtype=float)).all():
        raise ValueError("빈칸을 채운 뒤의 특성값은 모두 유한해야 합니다.")
    distances = {}
    for label, center in centers.iterrows():
        differences = completed - center
        distances[label] = (differences ** 2).sum(axis=1)
    return pd.DataFrame(distances, index=rows.index)

fill_values, centers = training_averages(known_train, feature_cols)
example = pd.DataFrame([[6, 3]], columns=feature_cols)
example_distances = squared_distances(example, fill_values, centers)
print("빈칸에 쓸 훈련 평균:", fill_values.to_numpy())
for label, center in centers.iterrows():
    distance = example_distances.at[0, label]
    print(f"라벨 {label}의 중심 {center.to_numpy()}, [6, 3]까지의 제곱거리 {distance:g}")


빈칸에 쓸 훈련 평균: [5. 3.]
라벨 -1의 중심 [2. 2.], [6, 3]까지의 제곱거리 17
라벨 1의 중심 [8. 4.], [6, 3]까지의 제곱거리 5


### 중심과 결정 경계

두 중심까지의 거리가 같은 위치를 **결정 경계**라고 합니다. 이 예에서는 특성이 [x,y]인 그림의 두 제곱거리가 (x−2)²+(y−2)²와 (x−8)²+(y−4)²입니다. 두 값이 같다는 식을 정리하면 3x+y=18입니다. [5.25,3]에서는 3×5.25+3=18.75이고 진품 중심 쪽이 더 가깝습니다.

T4의 첫째 특성만 9에서 11로 바꾼 훈련 표에서는 진품 중심이 [9,4]로 이동합니다. 이번에는 같은 [5.25,3]이 어느 중심에 더 가까운지 두 제곱거리를 비교하세요. 코드에서는 바뀐 훈련 표로 평균을 다시 구합니다. 거리 표에서 <code>idxmin(axis=1)</code>은 각 그림에 대해 가장 작은 제곱거리가 있는 열의 이름, 곧 예측 라벨을 반환합니다. 두 거리가 같으면 이 연습에서는 열을 라벨이 작은 순서로 두고 첫째 라벨을 고릅니다.


In [3]:
changed_train = known_train.copy()
changed_train.loc[changed_train["row_id"] == "T4", "feature1"] = 11
changed_fill, changed_centers = training_averages(changed_train, feature_cols)
print("원래 진품 중심:", centers.loc[1].to_numpy())
print("T4를 바꾼 뒤의 진품 중심:", changed_centers.loc[1].to_numpy())
print("원래 모델의 훈련 평균:", fill_values.to_numpy())

probe = pd.DataFrame([[5.25, 3]], columns=feature_cols)
original_probe = squared_distances(probe, fill_values, centers).idxmin(axis=1).iloc[0]
changed_probe = squared_distances(probe, changed_fill, changed_centers).idxmin(axis=1).iloc[0]
print("[5.25, 3]의 예측 (원래 모델 / T4를 바꾼 모델):", original_probe, "/", changed_probe)
assert np.allclose(changed_centers.loc[1], [9, 4])
assert np.allclose(centers.loc[1], [8, 4])
assert (original_probe, changed_probe) == (1, -1)


원래 진품 중심: [8. 4.]
T4를 바꾼 뒤의 진품 중심: [9. 4.]
원래 모델의 훈련 평균: [5. 3.]
[5.25, 3]의 예측 (원래 모델 / T4를 바꾼 모델): 1 / -1


## 3. 검증 정확도와 데이터 누수

다음 네 그림은 훈련에 사용하지 않은 연습용 검증 표본입니다. V1~V4는 이 표에서 그림을 구별하는 식별자입니다. V3의 특성은 방금 손으로 계산한 [6,3]이고 V4의 첫째 특성은 비어 있습니다. V4에는 어느 훈련 평균을 적용해야 할까요?

각 검증 그림의 제곱거리를 계산할 때는 훈련 그림에서 이미 구한 열 평균과 두 클래스 중심을 사용합니다. 예측 라벨은 두 제곱거리 중 더 작은 값이 있는 열의 이름입니다.


In [4]:
validation = pd.DataFrame([
    ("V1", 2, 3), ("V2", 8, 3), ("V3", 6, 3), ("V4", np.nan, 5),
], columns=["row_id", "feature1", "feature2"])
validation_distances = squared_distances(validation, fill_values, centers)
val_predictions = validation_distances.idxmin(axis=1)
worked_first = validation[["row_id"]].copy()
worked_first["prediction"] = val_predictions
display(worked_first)


,row_id,prediction
0,V1,-1
1,V2,1
2,V3,1
3,V4,1


검증 예측을 저장했으므로 이제 정답과 비교할 수 있습니다. 정답 표의 행 순서가 예측 표와 다릅니다. 같은 <code>row_id</code>끼리 연결한 뒤 정확도, 즉 **맞힌 그림 수를 평가한 그림 수로 나눈 값**을 구하세요.


In [5]:
worked_validation_key = pd.DataFrame([
    ("V4", 1), ("V1", -1), ("V3", -1), ("V2", 1),
], columns=["row_id", "label"])
val_view = worked_first.merge(worked_validation_key, on="row_id", validate="one_to_one")
assert len(val_view) == len(worked_first) == len(worked_validation_key)
val_view["correct"] = val_view["prediction"] == val_view["label"]
accuracy = val_view["correct"].sum() / len(val_view)
display(val_view)
print("맞힌 그림 수 / 평가한 그림 수:", val_view["correct"].sum(), "/", len(val_view))


,row_id,prediction,label,correct
0,V1,-1,-1,True
1,V2,1,1,True
2,V3,1,-1,False
3,V4,1,1,True


맞힌 그림 수 / 평가한 그림 수: 3 / 4


V3은 복제품이지만 모델은 진품으로 예측했습니다. 나머지 세 점을 맞혀 검증 정확도는 3/4입니다. 클래스의 중심이 그 클래스 안의 제곱거리 합을 최소로 만들어도, 모든 그림의 진위를 맞힌다는 보장은 없습니다.

훈련할 때 검증 정답을 사용하거나 검증값까지 섞어 대체 평균을 구하면 학습에 평가 데이터의 정보가 들어갑니다. 이를 **데이터 누수**라고 합니다. 검증 결과로 모델을 여러 번 고르면 검증 점수도 그 표본에 맞춰질 수 있습니다. 최종 평가는 따로 보관한 테스트 데이터로 합니다.

아래에서는 검증 그림 V1의 첫째 특성만 바꿉니다. 예측을 다시 해도 훈련 평균과 중심 두 개는 원래 값이어야 합니다.


In [6]:
before_fill = fill_values.copy()
before_centers = centers.copy()
changed_validation = validation.copy()
changed_validation.loc[changed_validation["row_id"] == "V1", "feature1"] = 2000
changed_predictions = squared_distances(changed_validation, fill_values, centers).idxmin(axis=1)
print("V1의 예측 (원래 값 / 첫째 특성을 바꾼 값):", val_predictions.iloc[0], "/", changed_predictions.iloc[0])
print("빈칸에 쓸 훈련 평균:", fill_values.to_numpy())
print("훈련 자료에서 정한 클래스 중심:\n", centers.to_numpy())
assert (val_predictions.iloc[0], changed_predictions.iloc[0]) == (-1, 1)
assert fill_values.equals(before_fill) and centers.equals(before_centers)


V1의 예측 (원래 값 / 첫째 특성을 바꾼 값): -1 / 1
빈칸에 쓸 훈련 평균: [5. 3.]
훈련 자료에서 정한 클래스 중심:
 [[2. 2.]
 [8. 4.]]


<a id="pipeline-device"></a>

## 4. 독립 연습 · 세 특성으로 그림 분류하기

이 절부터 바로 실행할 수 있습니다. 아래 세 표는 원본 대회 데이터와 별도로 만든 **새 연습 데이터**입니다. 각 행은 그림 한 점이고, <code>item</code>은 그 그림의 식별자(ID)입니다. R1~R5는 훈련 그림, V1~V3는 검증 그림입니다. J3, J1, J2는 추가로 예측할 그림을 입력 순서대로 표시한 ID입니다.

각 그림에서 예측 전에 얻는 수는 <code>feature1</code>, <code>feature2</code>, <code>feature3</code>입니다. 세 특성의 척도는 같습니다. 이 연습의 <code>label</code>은 **0=복제품, 1=진품, −1=미확인**입니다. <code>verified_label</code>은 그림의 진위가 확인된 뒤에 기록한 판정값입니다. 빈 특성값은 이 연습에서 학습 대상으로 삼은 그림 중 해당 값이 있는 그림의 평균으로 채웁니다.

1. 지도학습에 쓸 그림의 ID와 실제 입력 열을 고르세요. <code>label</code>에 0이 들어 있는 그림과 −1이 들어 있는 그림을 어떻게 다룰지, <code>verified_label</code>을 입력에 넣으면 왜 문제가 생기는지 설명하세요.
2. 열 순서를 <code>feature1, feature2, feature3</code>로 고정하세요. 빈칸을 채울 평균과 두 클래스의 평균 벡터를 손으로 구하세요. 검증 그림 V3에서 각 중심까지의 제곱거리를 계산해 더 가까운 클래스를 예측하세요.
3. 같은 훈련 표에서 클래스별 평균 벡터를 구하고 각 검증 그림과 두 중심 사이의 제곱거리를 계산하는 코드를 작성하세요. V3에 대한 코드의 두 거리와 손계산을 비교하세요. 정답을 열기 전에 세 검증 그림의 식별자와 첫 예측을 <code>first_predictions</code>라는 표에 <code>item,prediction</code> 두 열로 기록하세요. 추가 입력 J3, J1, J2도 같은 중심으로 분류하세요.


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

exercise_train = pd.DataFrame([
    ("R1",  2,     0, 0, 0, 0), ("R2", np.nan, 0, 0, 2, 2),
    ("R3",  4,     1, 1, 2, 6), ("R4",      6, 1, 1, 4, 8),
    ("R5",  5,    -1, np.nan, 2, 4),
], columns=["item", "feature2", "label", "verified_label", "feature3", "feature1"])
exercise_validation = pd.DataFrame([
    (1, "V1", 2, 3), (3, "V2", 6, 5), (1, "V3", 5, 4),
], columns=["feature3", "item", "feature1", "feature2"])
exercise_input = pd.DataFrame([
    (5, 7, "J3", 3), (3, 1, "J1", 1), (np.nan, 2, "J2", 1),
], columns=["feature2", "feature1", "item", "feature3"])

# 데이터 셀을 다시 실행해도 이미 기록한 첫 예측을 지우지 않습니다.
if "first_predictions" not in globals():
    first_predictions = None
if "recorded_first_predictions" not in globals():
    recorded_first_predictions = None
validation_key_path = Path("data/pipeline_device_validation.csv")
print("훈련 데이터")
display(exercise_train)
print("검증 입력")
display(exercise_validation)
print("추가 입력 (파일에 기록된 순서)")
display(exercise_input)


훈련 데이터


,item,feature2,label,verified_label,feature3,feature1
0,R1,2.0,0,0.0,0,0
1,R2,NaN,0,0.0,2,2
2,R3,4.0,1,1.0,2,6
3,R4,6.0,1,1.0,4,8
4,R5,5.0,-1,NaN,2,4


검증 입력


,feature3,item,feature1,feature2
0,1,V1,2,3
1,3,V2,6,5
2,1,V3,5,4


추가 입력 (파일에 기록된 순서)


,feature2,feature1,item,feature3
0,5.0,7,J3,3
1,3.0,1,J1,1
2,NaN,2,J2,1


**훈련 표본과 입력 열**  
사용한 그림 ID와 열 이름, <code>verified_label</code>을 제외한 이유를 적으세요.

**손계산**  
세 특성의 대체 평균, 복제품 중심, 진품 중심을 적으세요. V3의 두 제곱거리와 예측에 이르는 계산을 쓰세요.


In [8]:
# 훈련 평균과 두 클래스의 중심을 구한 뒤, 검증 그림과 추가 입력의 예측을 기록하세요.


### 검증 정답과 비교하기

첫 예측을 기록한 뒤 아래 셀을 실행하세요. 자습용 [검증 정답 CSV](data/pipeline_device_validation.csv)는 공개 파일입니다. 내려받아 <code>data</code> 폴더에 두거나 <code>validation_key_path</code>에 파일 경로를 넣으세요. 미리 답을 보았다면 풀이에 표시하세요. 이 파일에는 검증 그림의 정답만 있습니다.


In [9]:
def values_by_item(table, value_column, expected_items):
    if not isinstance(table, pd.DataFrame) or table.columns.tolist() != ["item", value_column]:
        raise ValueError(f"필요한 두 열: item, {value_column}")
    expected_items = list(expected_items)
    received_items = table["item"].tolist()
    for items in (expected_items, received_items):
        if not all(isinstance(item, str) and item.strip() for item in items):
            raise ValueError("각 그림에 비어 있지 않은 식별자가 필요합니다.")
        if len(items) != len(set(items)):
            raise ValueError("그림 식별자가 중복됐습니다.")
    if set(received_items) != set(expected_items):
        raise ValueError("검증 그림의 식별자를 각각 한 번씩 적으세요.")
    values = table.set_index("item")[value_column]
    for value in values:
        integer = isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_))
        if not integer or value not in (0, 1):
            raise ValueError("정수 0 또는 1을 써 주세요.")
    return tuple((item, int(values.at[item])) for item in expected_items)

validation_comparison = None
if first_predictions is None:
    print("먼저 검증 그림별 첫 예측을 first_predictions에 기록하세요.")
else:
    first_rows = values_by_item(first_predictions, "prediction", exercise_validation["item"])
    if recorded_first_predictions is None:
        recorded_first_predictions = first_rows
    elif dict(first_rows) != dict(recorded_first_predictions):
        raise ValueError("보관한 첫 예측과 값이 다릅니다. 바꾼 예측은 별도로 비교하세요.")
    if Path(validation_key_path).is_file():
        key_rows = pd.read_csv(validation_key_path, dtype={"item": str})
        key_by_item = dict(values_by_item(key_rows, "label", exercise_validation["item"]))
        validation_comparison = pd.DataFrame(recorded_first_predictions, columns=["item", "prediction"])
        validation_comparison["label"] = validation_comparison["item"].map(key_by_item)
        display(validation_comparison)
    else:
        print("첫 예측을 보관했습니다. 정답 파일을 내려받은 뒤 validation_key_path를 확인하고 다시 실행하세요.")


먼저 검증 그림별 첫 예측을 first_predictions에 기록하세요.


그림별 첫 예측과 정답에서 **맞힌 수 / 평가한 수**를 구하고, 틀린 그림이 있으면 그 ID를 적으세요. 라벨을 확인한 뒤 모델을 바꿨다면 바꾸기 전후의 결과를 구별해 두세요. [통합 교재의 연습 문제](week01_integrated.ipynb#week01-check) · [2주차 분할·전처리·평가](../week02/week02_prediction_evaluation.ipynb)


In [10]:
# 검증 정확도를 계산하고 잘못 분류한 그림의 ID를 구하세요.


### 라벨 기호가 바뀌면 무엇을 확인해야 할까?

Antique 원본의 0은 미확인이고 −1은 복제품입니다. 반면 4절 연습 표에서는 −1이 미확인이고 0이 복제품입니다. 4절에서 라벨이 0인 그림을 제외하면 복제품 중심을 계산할 훈련 그림이 사라집니다. 문제를 바꿔 풀 때는 각 기호의 뜻을 먼저 확인하세요. Antique 원문은 미확인 그림의 **특성값도** 학습에 활용하도록 요구합니다. 이번 연습에서 계산한 중심은 라벨이 확인된 그림으로만 구했습니다. [Antique 원문](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Antique/Antique.ipynb)


<a id="pipeline-submit"></a>

## 선택 실습 · 예측을 원본 파일의 순서로 저장하기

Antique 원문에서는 검증 그림마다 <code>submissionA.csv</code>, 테스트 그림마다 <code>submissionB.csv</code>에 해당 파일의 입력 순서대로 −1 또는 1을 기록합니다. 각 파일에 열 이름(헤더)과 식별자 열은 없습니다. [공식 제출 형식, 4절](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Antique/Antique.ipynb)

아래는 **행 순서만 연습하는 합성 예**입니다. 원본 입력 파일의 그림 순서는 A05, A08, A02입니다. 여기서 <strong>그림 식별자(ID)</strong>는 행의 위치가 바뀌어도 같은 그림을 찾도록 붙인 이름입니다. A05는 한 그림의 이름이고 숫자 5는 예측 라벨이 아닙니다.

| 계산한 순서 | 그림 ID | 예측 라벨 |
|---:|---|---:|
| 1 | A08 | 1 |
| 2 | A02 | −1 |
| 3 | A05 | −1 |

각 ID의 예측을 원본 입력 순서대로 적으세요. 이 표와 아래 코드만으로 확인할 수 있습니다.


In [11]:
predicted_rows = [("A08", 1), ("A02", -1), ("A05", -1)]
original_row_order = ["A05", "A08", "A02"]
by_id = dict(predicted_rows)  # 그림 ID로 그 그림의 예측을 찾습니다.
for row_id in original_row_order:
    print(row_id, "→", by_id[row_id])


A05 → -1
A08 → 1
A02 → -1


원본 순서 A05, A08, A02에 해당하는 값은 −1, 1, −1입니다. 계산한 순서대로 1, −1, −1을 기록하면 A05와 A08의 예측이 서로 바뀝니다. 두 파일 모두 허용된 값 세 개를 담고 있으므로 줄 수와 값의 범위만 검사해서는 이 오류를 찾을 수 없습니다. 원본에 별도의 ID 열이 없으면 표를 읽을 때 **원본 행 위치**를 보관해 두었다가 그 순서를 복원하면 됩니다. [통합 교재의 매칭 연습](week01_integrated.ipynb#week01-match)

4절의 추가 입력을 파일로 저장하려면 <code>audit.csv</code>의 첫 줄에 <code>item,signal</code>을 쓰세요. <code>item</code>은 그림의 ID, <code>signal</code>은 4절의 예측 라벨 0 또는 1입니다. J3, J1, J2 순서로 저장한 파일을 다시 열어 그림별 값을 확인하세요. 입력을 J1, J2, J3 순서로 바꿔 다시 실행했을 때 같은 그림의 예측이 유지되는지도 검사하세요.


<a id="pipeline-files"></a>

### 선택 실습 · 두 CSV가 들어 있는 ZIP 검사하기

1–3절에서 구한 훈련 평균과 두 클래스의 중심으로 두 합성 표 A와 B를 분류합니다. 처음 읽은 순서를 보관한 뒤 계산용 표를 정렬합니다. Python 함수는 특성값 하나가 비었으면 훈련 평균을 적용하고, 특성 열이 통째로 없거나 그림 ID가 중복되면 오류를 냅니다.


In [12]:
def is_missing_id(value):
    if not pd.api.types.is_scalar(value):
        raise ValueError("ID는 하나의 값이어야 함")
    return bool(pd.isna(value)) or (isinstance(value, str) and not value.strip())

def predict_pairs(frame, means, centers):
    required = ["row_id"] + feature_cols
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise ValueError("필수 열 없음: " + ", ".join(missing))
    if frame["row_id"].map(is_missing_id).any() or frame["row_id"].duplicated().any():
        raise ValueError("row_id 결측 또는 중복")
    values = frame[feature_cols].apply(pd.to_numeric, errors="raise")
    labels = squared_distances(values, means, centers).idxmin(axis=1)
    return list(zip(frame["row_id"].tolist(), labels.tolist()))

original_A = pd.DataFrame([("A05", 4, np.nan), ("A08", 8, 4), ("A02", 1, 2)],
                          columns=["row_id", "feature1", "feature2"])
original_B = pd.DataFrame([("B04", 1, 1), ("B09", 9, 4)],
                          columns=["row_id", "feature1", "feature2"])
order_A = original_A["row_id"].tolist()
order_B = original_B["row_id"].tolist()
A = original_A.sort_values("feature2", ascending=False).reset_index(drop=True)
B = original_B.sort_values("feature2", ascending=False).reset_index(drop=True)
pairs_A = predict_pairs(A, fill_values, centers)
pairs_B = predict_pairs(B, fill_values, centers)
print("계산 순서의 A 그림과 예측:", pairs_A)
print("원본 A 파일의 그림 순서:", order_A)


계산 순서의 A 그림과 예측: [('A08', 1), ('A02', -1), ('A05', -1)]
원본 A 파일의 그림 순서: ['A05', 'A08', 'A02']


In [13]:
def align_predictions(expected_ids, pairs):
    ids = list(expected_ids)
    pairs = list(pairs)
    if any(is_missing_id(row_id) for row_id in ids):
        raise ValueError("요구 ID 결측")
    if len(ids) != len(set(ids)):
        raise ValueError("요구 ID 중복")
    mapping = {}
    for row_id, prediction in pairs:
        if is_missing_id(row_id):
            raise ValueError("예측 ID 결측")
        if row_id in mapping:
            raise ValueError("예측 ID 중복")
        if isinstance(prediction, (bool, np.bool_)) or not isinstance(prediction, (int, np.integer)):
            raise ValueError("예측은 정수 -1 또는 1")
        if int(prediction) not in (-1, 1):
            raise ValueError("예측은 -1 또는 1")
        mapping[row_id] = int(prediction)
    if set(mapping) != set(ids):
        raise ValueError("요구 ID와 예측 ID가 다름")
    return [mapping[row_id] for row_id in ids]

ordered_A = align_predictions(order_A, pairs_A)
ordered_B = align_predictions(order_B, pairs_B)
naive_A = [label for _, label in pairs_A]
print("A 계산 순서로 적은 값:", naive_A)
print("A 원본 순서로 복원한 값:", ordered_A)
print("B 원본 순서로 복원한 값:", ordered_B)
assert ordered_A == [-1, 1, -1]
assert ordered_B == [-1, 1]
assert naive_A != ordered_A


A 계산 순서로 적은 값: [1, -1, -1]
A 원본 순서로 복원한 값: [-1, 1, -1]
B 원본 순서로 복원한 값: [-1, 1]


제출 파일 <code>submission.ipynb</code>가 만들어야 하는 압축 파일의 이름은 <code>submission.zip</code>입니다. 다음 코드는 같은 이름의 연습 파일을 임시 폴더에 저장하고 각 CSV를 다시 읽어 파일명, 행 수, 값, 원본 그림 순서를 검사합니다. ID가 없는 파일에서 라벨이 같은 두 행을 맞바꾸면 파일 내용만으로는 구별할 수 없습니다.


In [14]:
import csv
import io
import zipfile

def csv_text(labels):
    return "".join(f"{value}\n" for value in labels)

def archive_texts(text_A, text_B):
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("submissionA.csv", text_A)
        archive.writestr("submissionB.csv", text_B)
    return stream.getvalue()

def inspect_archive(blob, ids_A, preds_A, ids_B, preds_B):
    wanted = {"submissionA.csv": align_predictions(ids_A, preds_A),
              "submissionB.csv": align_predictions(ids_B, preds_B)}
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        names = archive.namelist()
        if len(names) != 2 or set(names) != set(wanted):
            raise ValueError("ZIP 파일명이 다르거나 중복됨")
        actual = {}
        for name, expected in wanted.items():
            rows = list(csv.reader(io.StringIO(archive.read(name).decode("utf-8"))))
            if any(len(row) != 1 or row[0] not in ("-1", "1") for row in rows):
                raise ValueError("헤더 없는 한 열의 -1 또는 1이 필요함")
            labels = [int(row[0]) for row in rows]
            if len(labels) != len(expected):
                raise ValueError("입력 행 수와 예측 행 수가 다름")
            if labels != expected:
                raise ValueError("저장한 ID 대응 순서가 다름")
            actual[name] = labels
    return actual

archive_bytes = archive_texts(csv_text(ordered_A), csv_text(ordered_B))


In [15]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="toy-submission-") as directory:
    practice_zip = Path(directory) / "submission.zip"
    practice_zip.write_bytes(archive_bytes)
    read_back = practice_zip.read_bytes()
    verified = inspect_archive(read_back, order_A, pairs_A, order_B, pairs_B)
    print("임시 작업 폴더에 저장한 파일:", practice_zip.name)
    print("파일을 다시 열어 확인한 두 CSV:", verified)
    with zipfile.ZipFile(practice_zip) as archive:
        print("A CSV 세 줄:", archive.read("submissionA.csv").decode("utf-8").splitlines())
    assert read_back == archive_bytes
# 블록이 끝나면 임시 파일은 삭제됩니다.
print("임시 파일 삭제 여부:", not practice_zip.exists())
assert not practice_zip.exists()


임시 작업 폴더에 저장한 파일: submission.zip
파일을 다시 열어 확인한 두 CSV: {'submissionA.csv': [-1, 1, -1], 'submissionB.csv': [-1, 1]}
A CSV 세 줄: ['-1', '1', '-1']
임시 파일 삭제 여부: True


다음은 A의 예측을 원본 순서로 복원하지 않고 저장한 경우입니다. 검사에서 어떤 조건을 어긴 것으로 나오는지 확인하세요.


In [16]:
wrong_order_zip = archive_texts(csv_text(naive_A), csv_text(ordered_B))
try:
    inspect_archive(wrong_order_zip, order_A, pairs_A, order_B, pairs_B)
except ValueError as error:
    print("원본 순서와 다른 파일:", error)
else:
    raise AssertionError("행 순서 오류를 발견하지 못했습니다.")


원본 순서와 다른 파일: 저장한 ID 대응 순서가 다름
